In [10]:
# ==============================================================================
# SCRIPT DE CLASSIFICAÇÃO DE ARQUIVOS IFC (VERSÃO FINAL)
# ==============================================================================

# --- 1. Importações Necessárias ---
import os
import pandas as pd
import ifcopenshell
import ifcopenshell.util.element
import joblib
import json
import numpy as np

# --- 2. Configuração: Caminhos dos Arquivos ---
# Altere este caminho para o novo arquivo IFC que você quer classificar
IFC_FILE_PATH = r"C:\Users\lucas.galicioli\Downloads\RÔGGA EMPREENDIMENTOS-BRUSQUE HOME CLUB-2025-10-16-13-35-31-469\PHN21043-CLI-EX-0002-BIM-TOR-GER-MODELO_DA_TORRE-R00.ifc"

# Caminhos para os artefatos do modelo (ajuste se necessário)
MODEL_PATH = 'ifc_classifier_disciplinas_v1.pkl'
ENCODER_PATH = 'label_encoder_disciplinas_v1.pkl'
COLUMNS_PATH = 'colunas_modelo_disciplinas.json' # <-- Certifique-se que este é o arquivo CORRETO gerado pelo treino
MEDIANS_PATH = 'medianas_treinamento.json'

# --- 3. Carregamento dos Artefatos do Modelo ---
print("Carregando artefatos do modelo treinado...")
try:
    modelo = joblib.load(MODEL_PATH)
    label_encoder = joblib.load(ENCODER_PATH)
    with open(COLUMNS_PATH, 'r', encoding='utf-8') as f: # Adicionado encoding='utf-8' por segurança
        colunas_do_modelo = json.load(f)
    with open(MEDIANS_PATH, 'r', encoding='utf-8') as f: # Adicionado encoding='utf-8' por segurança
        medianas_treinamento = json.load(f)
    print("-> Artefatos carregados com sucesso!")
except FileNotFoundError as e:
    print(f"ERRO CRÍTICO: Não foi possível encontrar um dos arquivos do modelo: {e}")
    exit()
except Exception as e:
    print(f"ERRO CRÍTICO ao carregar artefatos: {e}")
    exit()

# --- 4. Funções de Extração de Dados do IFC ---
def get_building_storey(element):
    try:
        spatial_container = ifcopenshell.util.element.get_container(element)
        if spatial_container and spatial_container.is_a('IfcBuildingStorey'):
            return spatial_container.Name
    except Exception:
        pass
    return None

def get_material_name(element):
    material = ifcopenshell.util.element.get_material(element)
    if not material: return None
    if hasattr(material, 'Name'): return material.Name
    elif hasattr(material, 'MaterialLayers'):
        layer_names = [
            layer.Material.Name for layer in material.MaterialLayers
            if hasattr(layer, 'Material') and hasattr(layer.Material, 'Name')
        ]
        return ', '.join(layer_names) if layer_names else None
    return None

def get_quantity_value_legacy(element, quantity_name):
    for definition in getattr(element, 'IsDefinedBy', []):
        if definition.is_a('IfcRelDefinesByProperties'):
            prop_set = definition.RelatingPropertyDefinition
            if prop_set.is_a('IfcElementQuantity'):
                for quantity in prop_set.Quantities:
                    if quantity.Name == quantity_name:
                        value_attribute = next((attr for attr in dir(quantity) if attr.endswith('Value')), None)
                        if value_attribute:
                            return getattr(quantity, value_attribute)
    return None

# --- 5. Processamento Principal: Extração e Classificação ---
try:
    # ETAPA A: Extrair dados do IFC para um DataFrame
    print(f"\nIniciando processamento do arquivo IFC: {os.path.basename(IFC_FILE_PATH)}...")
    ifc_file = ifcopenshell.open(IFC_FILE_PATH)
    products = ifc_file.by_type('IfcProduct')
    element_data = []

    for product in products:
        if product.is_a('IfcOpeningElement') or product.is_a('IfcVirtualElement'):
            continue

        psets = ifcopenshell.util.element.get_psets(product)
        rogga_pset = psets.get('PSET_RÔGGA', {})

        element_info = {
            'Class': product.is_a(),
            'PredefinedType': getattr(product, 'PredefinedType', None),
            'Name': getattr(product, 'Name', None),
            'BuildingStorey': get_building_storey(product),
            'Material': get_material_name(product),
            'PSET_RÔGGA.RÔGGA_SEÇÃO': rogga_pset.get('RÔGGA_SEÇÃO', None),
            'PSET_RÔGGA.RÔGGA_DESCRIÇÃO': rogga_pset.get('RÔGGA_DESCRIÇÃO', None),
            'Width': get_quantity_value_legacy(product, 'Width'),
            'Thickness': get_quantity_value_legacy(product, 'Thickness'),
            'Length': get_quantity_value_legacy(product, 'Length'),
            'Height': get_quantity_value_legacy(product, 'Height'),
            'FileName': os.path.basename(IFC_FILE_PATH),
            'GlobalId': product.GlobalId
        }
        element_data.append(element_info)

    df_ifc_data = pd.DataFrame(element_data)
    print(f"-> {len(df_ifc_data)} elementos extraídos do IFC.")

    if df_ifc_data.empty:
        print("AVISO: Nenhum elemento (válido para o modelo) foi extraído do IFC. O script será encerrado.")
        exit()

    # ETAPA B: Pré-processar o DataFrame para o modelo
    print("\nIniciando pré-processamento dos dados...")

    features_selecionadas = [
        'Class', 'PredefinedType', 'BuildingStorey', 'Material',
        'PSET_RÔGGA.RÔGGA_SEÇÃO', 'PSET_RÔGGA.RÔGGA_DESCRIÇÃO',
        'Width', 'Thickness', 'Length', 'Height'
    ]
    for col in features_selecionadas:
        if col not in df_ifc_data.columns:
            df_ifc_data[col] = None
    df_para_prever = df_ifc_data[features_selecionadas].copy()

    # Trata nulos categóricos
    cat_cols = df_para_prever.select_dtypes(include=['object']).columns
    df_para_prever.loc[:, cat_cols] = df_para_prever.loc[:, cat_cols].fillna('Desconhecido')

    # Trata nulos numéricos com medianas do treino
    for col, mediana in medianas_treinamento.items():
        if col in df_para_prever.columns:
             df_para_prever.loc[:, col] = df_para_prever.loc[:, col].fillna(mediana)

    # Aplica One-Hot Encoding
    df_encodado = pd.get_dummies(df_para_prever)

    # Limpa nomes das colunas (DEVE SER IDÊNTICO AO TREINO)
    df_encodado.columns = df_encodado.columns.str.replace(r'\[|\]|<', '_', regex=True)

    # Alinha as colunas com o modelo usando o JSON correto
    df_final = df_encodado.reindex(columns=colunas_do_modelo, fill_value=0)
    print("-> Pré-processamento concluído.")

    # ETAPA C: Fazer as previsões
    print("\nRealizando previsões com o modelo...")
    previsoes_numericas = modelo.predict(df_final)
    previsoes_texto = label_encoder.inverse_transform(previsoes_numericas)
    print("-> Previsões realizadas com sucesso!")

    # ETAPA D: Apresentar o resultado
    df_ifc_data['Ô_CLS_CLASSIFICAÇÃO_SOLIBRI'] = previsoes_texto # Adiciona a coluna de previsão

    print("\n--- AMOSTRA DO RESULTADO DA CLASSIFICAÇÃO ---")
    colunas_para_mostrar = ['Class', 'Name', 'PredefinedType', 'Ô_CLS_CLASSIFICAÇÃO_SOLIBRI']
    colunas_existentes_para_mostrar = [col for col in colunas_para_mostrar if col in df_ifc_data.columns]
    print(df_ifc_data[colunas_existentes_para_mostrar].head(20).fillna(''))

    # ETAPA E: Salvar o resultado completo em um novo arquivo CSV
    base_ifc_name = os.path.splitext(os.path.basename(IFC_FILE_PATH))[0]
    output_filename = f"classificado_{base_ifc_name}.csv"
    # Salva na mesma pasta do script por padrão
    df_ifc_data.to_csv(output_filename, index=False, sep=';', decimal=',', encoding='utf-8-sig')
    print(f"\nResultado completo salvo em: {os.path.abspath(output_filename)}")

except FileNotFoundError:
    print(f"ERRO: O arquivo IFC não foi encontrado em: {IFC_FILE_PATH}")
except ValueError as ve:
     print(f"Ocorreu um erro de valor (pode ser feature_names mismatch se o JSON ainda estiver errado): {ve}")
     import traceback
     traceback.print_exc() # Mostra mais detalhes do erro
except Exception as e:
    print(f"Ocorreu um erro inesperado durante o processamento: {e}")
    import traceback
    traceback.print_exc() # Mostra mais detalhes do erro

Carregando artefatos do modelo treinado...
-> Artefatos carregados com sucesso!

Iniciando processamento do arquivo IFC: PHN21043-CLI-EX-0002-BIM-TOR-GER-MODELO_DA_TORRE-R00.ifc...
-> 22109 elementos extraídos do IFC.

Iniciando pré-processamento dos dados...
-> Pré-processamento concluído.

Realizando previsões com o modelo...
-> Previsões realizadas com sucesso!

--- AMOSTRA DO RESULTADO DA CLASSIFICAÇÃO ---
                      Class  \
0   IfcBuildingElementProxy   
1   IfcBuildingElementProxy   
2   IfcBuildingElementProxy   
3   IfcBuildingElementProxy   
4   IfcBuildingElementProxy   
5   IfcBuildingElementProxy   
6   IfcBuildingElementProxy   
7   IfcBuildingElementProxy   
8   IfcBuildingElementProxy   
9   IfcBuildingElementProxy   
10  IfcBuildingElementProxy   
11  IfcBuildingElementProxy   
12  IfcBuildingElementProxy   
13  IfcBuildingElementProxy   
14  IfcBuildingElementProxy   
15  IfcBuildingElementProxy   
16  IfcBuildingElementProxy   
17  IfcBuildingElementProxy 

In [11]:
# Tente carregar o arquivo especificando sep=';'
df_resultado = pd.read_csv(
    r"c:\Users\lucas.galicioli\ifc-classifier\notebooks\classificado_PHN21043-CLI-EX-0002-BIM-TOR-GER-MODELO_DA_TORRE-R00.csv",
    sep=';'
)
display(df_resultado)

,Class,PredefinedType,Name,BuildingStorey,Material,PSET_RÔGGA.RÔGGA_SEÇÃO,PSET_RÔGGA.RÔGGA_DESCRIÇÃO,Width,Thickness,Length,Height,FileName,GlobalId,Ô_CLS_CLASSIFICAÇÃO_SOLIBRI
0,IfcBuildingElementProxy,NOTDEFINED,Identificador de Origem1:Identificador de Orig...,01.1 TER,<Unnamed>,NaN,NaN,NaN,NaN,NaN,NaN,PHN21043-CLI-EX-0002-BIM-TOR-GER-MODELO_DA_TOR...,3zo9X6EYvA$v1Kfg_uJf4V,"Bombas de recalque, de drenagem pluvial"
1,IfcBuildingElementProxy,NOTDEFINED,ARC - SUPORTE PARA CONDENSADORAS:SUPORTE PARA ...,34. TPA,Steel,--,ROG_SUP_BASE PARA CONDENSADORAS,NaN,NaN,NaN,NaN,PHN21043-CLI-EX-0002-BIM-TOR-GER-MODELO_DA_TOR...,2RHGRx1Qr3VOG8Ua072XdS,"Bombas de recalque, de drenagem pluvial"
2,IfcBuildingElementProxy,NOTDEFINED,ARC - SUPORTE PARA CONDENSADORAS:SUPORTE E BAS...,34. TPA,Steel,--,ROG_SUP_BASE PARA CONDENSADORAS,NaN,NaN,NaN,NaN,PHN21043-CLI-EX-0002-BIM-TOR-GER-MODELO_DA_TOR...,2RHGRx1Qr3VOG8Ua072XYl,"Bombas de recalque, de drenagem pluvial"
3,IfcBuildingElementProxy,NOTDEFINED,ARC - SUPORTE PARA CONDENSADORAS:SUPORTE PARA ...,34. TPA,Steel,--,ROG_SUP_BASE PARA CONDENSADORAS,NaN,NaN,NaN,NaN,PHN21043-CLI-EX-0002-BIM-TOR-GER-MODELO_DA_TOR...,2RHGRx1Qr3VOG8Ua072XkR,"Bombas de recalque, de drenagem pluvial"
4,IfcBuildingElementProxy,NOTDEFINED,ARC - SUPORTE PARA CONDENSADORAS:SUPORTE E BAS...,34. TPA,Steel,--,ROG_SUP_BASE PARA CONDENSADORAS,NaN,NaN,NaN,NaN,PHN21043-CLI-EX-0002-BIM-TOR-GER-MODELO_DA_TOR...,2RHGRx1Qr3VOG8Ua072XkO,"Bombas de recalque, de drenagem pluvial"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22104,IfcBuildingStorey,NaN,35. TP1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PHN21043-CLI-EX-0002-BIM-TOR-GER-MODELO_DA_TOR...,005ngX3ZH5qBwVQW8DIxXx,"Sanca, cortineiro e testeira"
22105,IfcBuildingStorey,NaN,36. TAP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PHN21043-CLI-EX-0002-BIM-TOR-GER-MODELO_DA_TOR...,005ngX3ZH5qBwVQW8DIxXw,"Sanca, cortineiro e testeira"
22106,IfcBuildingStorey,NaN,37. TP1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PHN21043-CLI-EX-0002-BIM-TOR-GER-MODELO_DA_TOR...,005ngX3ZH5qBwVQW8DIxX5,"Sanca, cortineiro e testeira"
22107,IfcBuildingStorey,NaN,RSV,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PHN21043-CLI-EX-0002-BIM-TOR-GER-MODELO_DA_TOR...,005ngX3ZH5qBwVQW8DIxX4,"Sanca, cortineiro e testeira"


In [ ]:
import ifcopenshell
import ifcopenshell.util.element
import pandas as pd
import os

# --- Caminho do arquivo IFC ---
ifc_file_path = r"C:\Users\lucas.galicioli\Downloads\RÔGGA EMPREENDIMENTOS-BRUSQUE HOME CLUB-2025-10-16-13-35-31-469\PHN21043-ARQ-EX-0002-BIM-TOR-GER-R01.ifc"


# --- Caminho da sua matriz de classificação ---
# --- NOVO ---
path_to_matrix = r'C:\Users\lucas.galicioli\ifc-classifier\data\interim\classification-matrix.xlsx'
coluna_matrix_filename = 'FileName'            # Nome da coluna de arquivos na matriz
coluna_matrix_disciplina = 'Ô_CLS_DISCIPLINAS' # Nome da coluna de disciplina na matriz


# --- Funções Auxiliares ---

def get_building_storey(element):
    """ Encontra o IfcBuildingStorey no qual o elemento está contido. """
    try:
        spatial_container = ifcopenshell.util.element.get_container(element)
        if spatial_container and spatial_container.is_a('IfcBuildingStorey'):
            return spatial_container.Name
    except Exception:
        pass
    return None

def get_material_name(element):
    """ Extrai o nome do material associado ao elemento. """
    material = ifcopenshell.util.element.get_material(element)
    if not material:
        return None
    if hasattr(material, 'Name'):
        return material.Name
    elif hasattr(material, 'MaterialLayers'):
        layer_names = [
            layer.Material.Name 
            for layer in material.MaterialLayers 
            if hasattr(layer, 'Material') and hasattr(layer.Material, 'Name')
        ]
        return ', '.join(layer_names) if layer_names else None
    return None
    
def get_quantity_value_legacy(element, quantity_name):
    """
    Busca por uma quantidade específica (ex: 'Width') e retorna seu valor.
    """
    for definition in getattr(element, 'IsDefinedBy', []):
        if definition.is_a('IfcRelDefinesByProperties'):
            prop_set = definition.RelatingPropertyDefinition
            if prop_set.is_a('IfcElementQuantity'):
                for quantity in prop_set.Quantities:
                    if quantity.Name == quantity_name:
                        value_attribute = next((attr for attr in dir(quantity) if attr.endswith('Value')), None)
                        if value_attribute:
                            return getattr(quantity, value_attribute)
    return None

# --- NOVAS FUNÇÕES AUXILIARES PARA MAPEAMENTO ---

def gerar_mapa_disciplinas(matrix_path, col_filename, col_disciplina):
    """
    Carrega a matriz de classificação e cria um dicionário de mapeamento
    (Código -> Disciplina). Ex: {'HID': 'Hidrossanitário', 'EST': 'Estrutura'}
    """
    try:
        df_matrix = pd.read_excel(matrix_path)
        
        # Criar df temporário apenas com as colunas necessárias
        df_mapa_temp = df_matrix[[col_filename, col_disciplina]].copy()
        
        # Extrair o código (ex: "EST", "HID")
        df_mapa_temp['Disciplina_Code'] = df_mapa_temp[col_filename].str.split('-').str[1]
        
        # Limpar (remover nulos e duplicatas)
        df_mapa_temp = df_mapa_temp[['Disciplina_Code', col_disciplina]].dropna().drop_duplicates()
        
        # Converter para dicionário
        mapa = df_mapa_temp.set_index('Disciplina_Code')[col_disciplina].to_dict()
        
        if not mapa:
            print(f"AVISO: O mapa de disciplinas gerado a partir de '{matrix_path}' está vazio.")
        
        return mapa
    
    except FileNotFoundError:
        print(f"ERRO: Arquivo da matriz não encontrado em: {matrix_path}")
        return None
    except KeyError as e:
        print(f"ERRO: Coluna {e} não encontrada na matriz. Verifique os nomes '{col_filename}' e '{col_disciplina}'.")
        return None
    except Exception as e:
        print(f"ERRO ao gerar mapa de disciplinas: {e}")
        return None

def extrair_disciplina_do_nome(nome_arquivo_base, mapa_disciplinas):
    """
    Extrai o código do nome do arquivo e o traduz usando o mapa.
    """
    try:
        nome_base = os.path.splitext(nome_arquivo_base)[0]
        partes_nome = nome_base.split('-')
        codigo_disciplina = partes_nome[1] # Pega o "HID", "EST", etc.
        
        # Traduz usando o mapa
        disciplina_traduzida = mapa_disciplinas.get(codigo_disciplina, f"Código '{codigo_disciplina}' Não Mapeado")
        return disciplina_traduzida
    except IndexError:
        print(f"AVISO: O nome '{nome_arquivo_base}' não segue o padrão 'XXX-CODIGO-...'")
        return "Erro: Padrão de Nome"
    except Exception:
        return "Erro: Extração"

# --- Processamento Principal ---

try:
    ifc_file = ifcopenshell.open(ifc_file_path)
    file_name = os.path.basename(ifc_file_path)

    element_data = []
    products = ifc_file.by_type('IfcProduct')

    print(f"Processando {len(products)} elementos do arquivo: {file_name}...")

    for product in products:
        if product.is_a('IfcOpeningElement') or product.is_a('IfcVirtualElement'):
            continue

        psets = ifcopenshell.util.element.get_psets(product)
        rogga_pset = psets.get('PSET_RÔGGA', {})

        element_info = {
            'GlobalId': product.GlobalId,
            'FileName': file_name,
            'Class': product.is_a(),
            'PredefinedType': getattr(product, 'PredefinedType', None),
            'Name': getattr(product, 'Name', None),
            'BuildingStorey': get_building_storey(product),
            'Material': get_material_name(product),
            'PSET_RÔGGA.RÔGGA_SEÇÃO': rogga_pset.get('RÔGGA_SEÇÃO', None),
            'PSET_RÔGGA.RÔGGA_DESCRIÇÃO': rogga_pset.get('RÔGGA_DESCRIÇÃO', None),
            'Width': get_quantity_value_legacy(product, 'Width'),
            'Thickness': get_quantity_value_legacy(product, 'Thickness'),
            'Length': get_quantity_value_legacy(product, 'Length'),
            'Height': get_quantity_value_legacy(product, 'Height'),
        }
        
        element_data.append(element_info)

    df_ifc_data = pd.DataFrame(element_data)

    # --- INÍCIO DA LÓGICA DE CLASSIFICAÇÃO DE DISCIPLINA ---
    # --- NOVO ---
    print("\nDataFrame criado. Gerando mapa e adicionando classificação de Disciplina...")

    # 1. Gerar o mapa de disciplinas (carregando a matriz)
    mapa_disciplinas = gerar_mapa_disciplinas(path_to_matrix, coluna_matrix_filename, coluna_matrix_disciplina)

    # 2. Extrair e traduzir a disciplina do arquivo atual
    disciplina_final = "Erro ao Mapear" # Valor padrão
    if mapa_disciplinas:
        # Usa a variável 'file_name' que já definimos no começo do 'try'
        disciplina_final = extrair_disciplina_do_nome(file_name, mapa_disciplinas)
    else:
        print("AVISO: Não foi possível carregar o mapa. A disciplina não será adicionada corretamente.")

    # 3. Adicionar a coluna ao DataFrame
    #    (O mesmo valor será aplicado a todas as linhas, o que está correto)
    df_ifc_data['Ô_CLS_DISCIPLINAS'] = disciplina_final
    print(f"Ô_CLS_DISCIPLINAS '{disciplina_final}' atribuída a {len(df_ifc_data)} elementos.")
    
    # --- FIM DA LÓGICA DE CLASSIFICAÇÃO ---


    # --- Reordenação das Colunas ---
    desired_order = [
        'Class',
        'PredefinedType', 'BuildingStorey', 'Material', 'Name',
        'PSET_RÔGGA.RÔGGA_SEÇÃO', 'PSET_RÔGGA.RÔGGA_DESCRIÇÃO',
        'Width', 'Thickness', 'Length', 'Height',
        'FileName',
        'Ô_CLS_DISCIPLINAS', # Coluna nova adicionada à lista
        'GlobalId'
    ]

    # Garante que todas as colunas existem (bom para robustez)
    for col in desired_order:
        if col not in df_ifc_data.columns:
            df_ifc_data[col] = None
            
    df_ifc_data = df_ifc_data[desired_order]

    print("\nDataset criado com sucesso! Amostra dos dados (com Ô_CLS_DISCIPLINAS):")
    display(df_ifc_data)

except FileNotFoundError:
    print(f"ERRO: O arquivo não foi encontrado em: {ifc_file_path}")
except Exception as e:
    print(f"Ocorreu um erro inesperado: {e}")

Ocorreu um erro inesperado: argument should be a str or an os.PathLike object where __fspath__ returns a str, not 'DataFrame'


In [ ]:
# -*- coding: utf-8 -*-

import ifcopenshell
from ifcopenshell import api
from ifcopenshell.util import element
import os

# --- PASSO 1: CARREGAR O ARQUIVO IFC ---
ifc_file_path = r'C:\Users\lucas.galicioli\ifc-classifier\data\raw\PHN21021A-HID-LO-0002-BIM-T01-GER-TORRE 01-R00.ifc'#<-- AJUSTE O CAMINHO AQUI

if not os.path.exists(ifc_file_path):
    print(f"ERRO: O arquivo não foi encontrado em: {ifc_file_path}")
else:
    ifc_file = ifcopenshell.open(ifc_file_path)
    print(f"Arquivo '{os.path.basename(ifc_file_path)}' carregado com sucesso!")

    # --- PASSO 2: DEFINA O NOME DO PSET E AS PROPRIEDADES ---
    nome_do_pset = "RÖGGA_CLASSIFICAÇÃO"
    novas_propriedades = {
        "RÖGGA_CLS_SISTEMA": "HID-AGUA-FRIA",
        "RÖGGA_CLS_STATUS": "Verificado"
    }

    # --- PASSO 3: SELECIONE OS ELEMENTOS ALVO E APLIQUE O PSET ---
    elementos_alvo = ifc_file.by_type('IfcValve')
    
    if not elementos_alvo:
        print("\nAVISO: Nenhum elemento do tipo 'IfcValve' foi encontrado no modelo.")
    else:
        print(f"\nEncontrados {len(elementos_alvo)} elementos 'IfcValve'. Adicionando o Pset a cada um...")

        # Loop para aplicar o Pset a cada elemento
        for alvo in elementos_alvo:
            # ETAPA 3.1: Criar o Pset vazio e obter a referência para ele
            pset = ifcopenshell.api.run("pset.add_pset", 
                                        ifc_file, 
                                        product=alvo, 
                                        name=nome_do_pset)
            
            # ETAPA 3.2: Editar o Pset recém-criado para adicionar as propriedades
            ifcopenshell.api.run("pset.edit_pset", 
                                 ifc_file, 
                                 pset=pset, 
                                 properties=novas_propriedades)

        print(f"Pset '{nome_do_pset}' foi adicionado com sucesso a {len(elementos_alvo)} elementos.")

        # --- PASSO 4: VERIFIQUE O RESULTADO (em um elemento de amostra) ---
        print("\n--- Verificando o primeiro elemento modificado para confirmar a alteração: ---")
        primeiro_alvo_modificado = elementos_alvo[0]
        print(f"Elemento de amostra: {primeiro_alvo_modificado.Name} (ID: {primeiro_alvo_modificado.id()})")
        
        todos_os_psets_modificados = element.get_psets(primeiro_alvo_modificado, psets_only=True)

        for pset_name, properties in todos_os_psets_modificados.items():
            print(f"\n  -> Pset: '{pset_name}'")
            for prop_name, prop_value in properties.items():
                print(f"     - {prop_name}: {prop_value}")


        # --- PASSO 5: SALVE O ARQUIVO IFC MODIFICADO ---
        output_dir = 'data/processed'
        os.makedirs(output_dir, exist_ok=True)
        
        nome_arquivo_original = os.path.basename(ifc_file_path)
        novo_nome_arquivo = f"{os.path.splitext(nome_arquivo_original)[0]}_modificado.ifc"
        caminho_novo_arquivo = os.path.join(output_dir, novo_nome_arquivo)

        ifc_file.write(caminho_novo_arquivo)

        print(f"\nArquivo IFC modificado foi salvo em: {caminho_novo_arquivo}")
        